# Baseline Experiments
Runs **Vanilla LLM** and **RAG-only** baselines for NomadBench,
then prints the comparison table (Comment 2) and the Tier-1 per-task
breakdown (Comment 3).

**Run order:** Cell 1 (imports) → Cell 2 (Vanilla) → Cell 3 (RAG) → Cell 4 (table) → Cell 5 (T1 breakdown)

In [1]:
import sys, os, importlib, json, time, traceback
from pathlib import Path
from typing import Dict, List, Optional

# ── Path setup (same pattern as agent_test.ipynb) ────────────────────────
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()   # Jupyter: use kernel working directory

# nomad/src  (agents, tools, config, llm, state)
sys.path.insert(0, notebook_dir)
# nomad/benchmark/src  (run_benchmark, statistics)
sys.path.insert(0, os.path.join(notebook_dir, "..", "benchmark", "src"))

# Force-reload so edits are picked up without kernel restart
import llm, agents.specialist, agents.orchestrator, agents.verifier, state
for _m in [llm, state, agents.specialist, agents.orchestrator, agents.verifier]:
    importlib.reload(_m)

from agents.orchestrator import analyze_user_input, update_state_from_analysis
from agents.specialist   import (
    search_flight_candidates, search_hotel_candidates,
    search_activity_candidates, select_top_k,
)
from llm   import call_llm_structured, set_llm_task_id
from state import TravelState
from tools.plan_repository import PlanRepository
from tools.evaluator       import NomadEvaluator
from config import OUTPUT_DIR

import run_benchmark
importlib.reload(run_benchmark)
from run_benchmark import (
    load_orchestrator_tasks, filter_tasks, _group_multi_turn_tasks,
)

# ── Output directories ────────────────────────────────────────────────────
PLANS_VANILLA_DIR = os.path.join(OUTPUT_DIR, "plans_vanilla")
PLANS_RAG_DIR     = os.path.join(OUTPUT_DIR, "plans_rag")
EVAL_DIR          = os.path.join(OUTPUT_DIR, "evaluations")
TASKS_FILE        = os.path.join(notebook_dir, "..", "benchmark", "data", "orchestrator_tasks.json")
for _d in [PLANS_VANILLA_DIR, PLANS_RAG_DIR, EVAL_DIR]:
    os.makedirs(_d, exist_ok=True)

print("✓ Imports OK")
print(f"  OUTPUT_DIR   = {OUTPUT_DIR}")
print(f"  TASKS_FILE   = {os.path.abspath(TASKS_FILE)}")

✓ Imports OK
  OUTPUT_DIR   = c:\GIT\CS498_Nomad_Traveller_Agent\nomad\src\..\output
  TASKS_FILE   = c:\GIT\CS498_Nomad_Traveller_Agent\nomad\benchmark\data\orchestrator_tasks.json


## Shared helpers

In [2]:
# ── Schema for both baselines (matches specialist.SELECTOR_SCHEMA) ────────
_PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "itinerary": {
            "type": "object",
            "properties": {
                "flights":              {"type": "object"},
                "hotels":               {"type": "object"},
                "activities":           {"type": "array", "items": {"type": "object"}},
                "estimated_total_cost": {"type": "number"},
            },
        },
        "constraints_met":       {"type": "boolean"},
        "unmet_constraints":     {"type": "array", "items": {"type": "string"}},
        "reasoning":             {"type": "string"},
        "final_message_to_user": {"type": "string"},
    },
    "required": ["itinerary", "constraints_met", "unmet_constraints",
                 "reasoning", "final_message_to_user"],
}

def _run_orchestrator(task, state):
    """Extract constraints via Orchestrator (shared by both baselines)."""
    user_query = task["input"].get("content", "")
    expected   = task.get("expected_output", {})
    task_json  = {
        "task_id":             task["task_id"],
        "updated_constraints": expected.get("updated_constraints", {}),
    }
    analysis = analyze_user_input(user_msg=user_query, state=state, task_json=task_json)
    return update_state_from_analysis(state, analysis)

def _mean(vals):
    return sum(vals) / len(vals) if vals else 0.0

print("✓ Helpers defined")

✓ Helpers defined


---
## Cell 2 — Vanilla LLM baseline
Orchestrator extracts constraints, then Claude generates the itinerary from
**parametric memory only** — no SerpAPI calls.

In [ ]:
_VANILLA_SYSTEM = """You are a travel planning assistant.
Generate a complete, realistic travel itinerary based ONLY on your training knowledge.
Do NOT mention that you are fabricating data — produce the best plan you can.
Fill all cost fields with plausible USD numbers."""


def run_vanilla_task(task, prior_state=None, verbose=True):
    task_id    = task["task_id"]
    user_query = task["input"].get("content", "")
    t0         = time.time()
    try:
        state = prior_state if prior_state else TravelState(task_id=task_id)
        state.task_id = task_id
        set_llm_task_id(task_id)

        state = _run_orchestrator(task, state)
        constraints_str = state.constraints.model_dump_json(indent=2)

        messages = [{"role": "user", "content": (
            f"Plan this trip using your knowledge:\n\n"
            f"REQUEST: {user_query}\n\nCONSTRAINTS:\n{constraints_str}\n\n"
            f"Produce a complete itinerary with realistic flights, hotels, "
            f"activities, and total cost."
        )}]
        plan = call_llm_structured(
            messages=messages, schema=_PLAN_SCHEMA, system=_VANILLA_SYSTEM
        )
        PlanRepository(base_dir=PLANS_VANILLA_DIR).save_plan(
            plan=plan, task_id=task_id, save_metadata=True
        )
        if verbose:
            cost = plan.get("itinerary", {}).get("estimated_total_cost", "?")
            print(f"  ✓ {task_id}  cost=${cost}  "
                  f"constraints_met={plan.get('constraints_met')}  "
                  f"({round(time.time()-t0,1)}s)")
        return {"status": "success", "final_state": state}
    except Exception as e:
        print(f"  ✗ {task_id}  ERROR: {e}")
        traceback.print_exc()
        return {"status": "error", "error": str(e), "final_state": prior_state}
    finally:
        set_llm_task_id(None)


# ── Run all tasks ─────────────────────────────────────────────────────────
# Change tier= or task_id= to run a subset, e.g. tier=1 or task_id="ORCH-T1-03"
tasks  = load_orchestrator_tasks(TASKS_FILE)
tasks  = filter_tasks(tasks)   # all tasks; change to filter_tasks(tasks, tier=1) for T1 only
groups = _group_multi_turn_tasks(tasks)

print(f"Running Vanilla LLM on {sum(len(g) for g in groups)} tasks...\n")
vanilla_run_results = []
for group in groups:
    carry = None
    for task in group:
        r = run_vanilla_task(task, prior_state=carry)
        carry = r.get("final_state")
        vanilla_run_results.append(r)

ok = sum(1 for r in vanilla_run_results if r["status"] == "success")
print(f"\nDone: {ok}/{len(vanilla_run_results)} success")

Running Vanilla LLM on 32 tasks...

  [LLM CACHE SAVE] cc1a081295621be2
  [LLM CACHE SAVE] 5b1e691b5a424059
✓ Plan saved: c:\GIT\CS498_Nomad_Traveller_Agent\nomad\src\..\output\plans_vanilla\ORCH-T1-01\plan.json
  ✓ ORCH-T1-01  cost=$2448  constraints_met=True  (33.7s)
  [LLM CACHE SAVE] 1909d79833ef80bb
  [LLM CACHE SAVE] 05d3edb930bef419
✓ Plan saved: c:\GIT\CS498_Nomad_Traveller_Agent\nomad\src\..\output\plans_vanilla\ORCH-T1-02\plan.json
  ✓ ORCH-T1-02  cost=$643.0  constraints_met=True  (32.8s)
  [LLM CACHE SAVE] 0dc874d8f69096bc


---
## Cell 3 — RAG-only baseline
Full search + select pipeline, but plan acceptance is driven by the **LLM
selector's own** `constraints_met` flag (no programmatic validation).
When the LLM reports failure, `closest_alternative` is saved — this
simulates false-positive budget violations.

In [ ]:
def run_rag_task(task, prior_state=None, verbose=True):
    task_id = task["task_id"]
    t0      = time.time()
    try:
        state = prior_state if prior_state else TravelState(task_id=task_id)
        state.task_id = task_id
        set_llm_task_id(task_id)

        state = _run_orchestrator(task, state)
        needs = state.needs

        if not (needs.flight or needs.hotel or needs.activity):
            print(f"  — {task_id}  skipped (no needs)")
            return {"status": "skipped", "final_state": state}

        # Same search as Nomad
        search_results = {"flights": [], "hotels": [], "activities": []}
        if needs.flight:   search_results["flights"]    = search_flight_candidates(state.constraints, task_id)
        if needs.hotel:    search_results["hotels"]     = search_hotel_candidates(state.constraints, task_id)
        if needs.activity: search_results["activities"] = search_activity_candidates(state.constraints, task_id)

        # Same LLM selection as Nomad
        selection = select_top_k(
            task_id=task_id,
            constraints_json=state.constraints.model_dump_json(indent=2),
            needs=needs,
            search_results=search_results,
            top_k=5,
        )

        # RAG-only: trust LLM's own constraints_met — no programmatic check
        if selection.get("constraints_met", True):
            plan_to_save = selection
        else:
            # LLM-reported failure → save closest_alternative (the degraded plan)
            alt = selection.get("closest_alternative")
            plan_to_save = dict(selection)
            plan_to_save["itinerary"] = alt if alt else selection.get("itinerary", {})
            if verbose:
                print(f"  ⚠ {task_id}  LLM false-positive → saving closest_alternative")

        PlanRepository(base_dir=PLANS_RAG_DIR).save_plan(
            plan=plan_to_save, task_id=task_id, save_metadata=True
        )
        if verbose:
            cost  = plan_to_save.get("itinerary", {}).get("estimated_total_cost", "?")
            unmet = selection.get("unmet_constraints", [])
            print(f"  ✓ {task_id}  cost=${cost}  "
                  f"LLM-met={selection.get('constraints_met')}  "
                  f"unmet={unmet}  ({round(time.time()-t0,1)}s)")
        return {"status": "success", "final_state": state}
    except Exception as e:
        print(f"  ✗ {task_id}  ERROR: {e}")
        traceback.print_exc()
        return {"status": "error", "error": str(e), "final_state": prior_state}
    finally:
        set_llm_task_id(None)


# ── Run all tasks ─────────────────────────────────────────────────────────
tasks  = load_orchestrator_tasks(TASKS_FILE)
tasks  = filter_tasks(tasks)
groups = _group_multi_turn_tasks(tasks)

print(f"Running RAG-only on {sum(len(g) for g in groups)} tasks...\n")
rag_run_results = []
for group in groups:
    carry = None
    for task in group:
        r = run_rag_task(task, prior_state=carry)
        carry = r.get("final_state")
        rag_run_results.append(r)

ok = sum(1 for r in rag_run_results if r["status"] == "success")
print(f"\nDone: {ok}/{len(rag_run_results)} success")

---
## Cell 4 — Evaluate & print comparison table

In [ ]:
def evaluate_baseline(plans_dir, force_zero_tool=False):
    """
    Evaluate all saved plans in plans_dir.
    force_zero_tool=True injects a dummy tool log so the evaluator
    cannot infer tool usage from plan content (used for Vanilla LLM).
    """
    repo      = PlanRepository(base_dir=plans_dir)
    task_ids  = repo.get_all_plans()
    evaluator = NomadEvaluator(
        task_file=TASKS_FILE if Path(TASKS_FILE).exists() else None
    )
    results = {}
    for tid in sorted(task_ids):
        try:
            plan      = repo.load_plan(tid)
            tool_logs = [{"tool": "__no_api_calls__"}] if force_zero_tool else []
            r         = evaluator.evaluate(agent_output=plan, task_id=tid, tool_logs=tool_logs)
            results[tid] = r
        except Exception as e:
            print(f"  [{tid}] eval error: {e}")
            results[tid] = {"error": str(e)}
    return results


def _agg(results):
    ok = {k: v for k, v in results.items() if "overall_score" in v}
    overall = [v["overall_score"] for v in ok.values()]
    csr     = [v["csr_score"]     for v in ok.values()]
    binary  = [1 if v["csr_score"] >= 1.0 else 0 for v in ok.values()]
    tool    = [v["tool_accuracy"] for v in ok.values()]
    inter   = [v["interest_score"] for v in ok.values() if v.get("interest_score") is not None]
    return {
        "overall":  _mean(overall), "csr_frac": _mean(csr),
        "csr_bin":  _mean(binary),  "tool":     _mean(tool),
        "interest": _mean(inter) if inter else None, "n": len(overall),
    }


# ── Evaluate ──────────────────────────────────────────────────────────────
print("Evaluating Vanilla LLM...")
vanilla_eval = evaluate_baseline(PLANS_VANILLA_DIR, force_zero_tool=True)

print("\nEvaluating RAG-only...")
rag_eval = evaluate_baseline(PLANS_RAG_DIR, force_zero_tool=False)

# Load existing Nomad results if available
nomad_eval = {}
nomad_report = Path(EVAL_DIR) / "benchmark_evaluation.json"
if nomad_report.exists():
    nomad_eval = json.loads(nomad_report.read_text()).get("results", {})
    print(f"\nLoaded Nomad results from {nomad_report}  ({len(nomad_eval)} tasks)")

# ── Print table ────────────────────────────────────────────────────────────
rows = [
    ("Vanilla LLM",  _agg(vanilla_eval)),
    ("RAG-only",     _agg(rag_eval)),
]
if nomad_eval:
    rows.append(("Nomad (ours)", _agg(nomad_eval)))

print()
print(f"{'System':<16}  {'Overall':>8}  {'CSR(frac)':>10}  {'CSR(bin)':>9}  {'Tool Acc':>9}  {'Interest':>9}  {'n':>4}")
print("-" * 72)
for name, a in rows:
    inter = f"{a['interest']:.1%}" if a["interest"] is not None else "  ---"
    print(f"{name:<16}  {a['overall']:>8.1%}  {a['csr_frac']:>10.1%}  "
          f"{a['csr_bin']:>9.1%}  {a['tool']:>9.1%}  {inter:>9}  {a['n']:>4}")
print("=" * 72)

# Save combined JSON
out = {"vanilla": vanilla_eval, "rag": rag_eval}
out_path = Path(EVAL_DIR) / "baseline_comparison.json"
out_path.write_text(json.dumps(out, indent=2, ensure_ascii=False))
print(f"\nSaved → {out_path}")

---
## Cell 5 — Tier-1 per-task breakdown
Shows individual metric scores and **which constraints failed** for every T1 task,
explaining why T1 mean < T3 mean.

In [ ]:
from config import PLANS_DIR   # Nomad's own plans (not the baselines)

def _constraint_label(c):
    cl = c.lower()
    if "budget" in cl or "$" in cl:              return "Budget"
    if "destination" in cl:                       return "Destination"
    if "origin" in cl or "departure" in cl:       return "Origin/Dep"
    if "hotel_location" in cl or "hotel location" in cl: return "Hotel location"
    if "start_date" in cl:                        return "Start date"
    if "end_date" in cl:                          return "End date"
    if "star" in cl or "rating" in cl:            return "Hotel star"
    if "per night" in cl or "nightly" in cl:      return "Hotel rate"
    if "vegetarian" in cl or "vegan" in cl:       return "Dietary"
    return c[:28]

def _failure_type(c):
    cl = c.lower()
    loc_kw = ["hotel_location","hotel location","near ","downtown","close to","south congress"]
    if any(k in cl for k in loc_kw):              return "Fine-grained geo"
    if ("star" in cl or "rating" in cl) and ("null" in cl or "none" in cl or ": 0" in cl):
        return "Metadata gap"
    if "hotel star" in cl or "star:" in cl:       return "Metadata gap"
    if "budget" in cl:                            return "Budget exceeded"
    if "airport" in cl:                           return "IATA mismatch"
    return "Constraint not met"

# ── Evaluate T1 tasks from Nomad's own plans ──────────────────────────────
tasks_raw  = json.loads(Path(TASKS_FILE).read_text())
t1_tasks   = {t["task_id"]: t for t in tasks_raw if t["tier"] == 1}
repo       = PlanRepository(base_dir=PLANS_DIR)
evaluator  = NomadEvaluator(task_file=TASKS_FILE)

breakdown_rows = []
for tid in sorted(t1_tasks):
    if not repo.plan_exists(tid):
        print(f"  [{tid}] no saved plan — run run_benchmark.py first")
        continue
    plan = repo.load_plan(tid)
    r    = evaluator.evaluate(agent_output=plan, task_id=tid)

    csr_frac    = r["csr_score"]
    csr_bin     = csr_frac >= 1.0
    consistency = 0.5 if r.get("conflict_report", {}).get("has_conflicts") else 1.0
    failing     = [c for c, met in r.get("constraint_breakdown", {}).items() if not met]
    ftypes      = [_failure_type(c) for c in failing]

    breakdown_rows.append({
        "task_id":     tid,
        "desc":        t1_tasks[tid].get("description", ""),
        "overall":     r["overall_score"],
        "schema":      r["schema_compliance"],
        "csr_frac":    csr_frac,
        "csr_bin":     csr_bin,
        "tool_acc":    r["tool_accuracy"],
        "consistency": consistency,
        "failing":     failing,
        "ftypes":      ftypes,
    })

# ── Print table ────────────────────────────────────────────────────────────
print(f"{'='*105}")
print("TIER-1 PER-TASK BREAKDOWN")
print(f"{'='*105}")
print(f"{'Task ID':<16}  {'Overall':>8}  {'Schema':>7}  {'CSR(fr)':>8}  "
      f"{'CSR(bin)':>9}  {'Tool':>6}  {'Cons':>6}  Failing constraints")
print("-" * 105)

for row in breakdown_rows:
    bin_str  = "✓ PASS" if row["csr_bin"] else "✗ FAIL"
    fail_str = "  —" if not row["failing"] else "; ".join(
        f"{_constraint_label(c)} [{ft}]"
        for c, ft in zip(row["failing"], row["ftypes"])
    )
    print(f"{row['task_id']:<16}  {row['overall']:>8.1%}  {row['schema']:>7.1%}  "
          f"{row['csr_frac']:>8.1%}  {bin_str:>9}  "
          f"{row['tool_acc']:>6.1%}  {row['consistency']:>6.1%}  {fail_str}")

# ── Aggregate ─────────────────────────────────────────────────────────────
print("-" * 105)
n = len(breakdown_rows)
if n:
    print(f"{'MEAN (n='+str(n)+')':<16}  "
          f"{_mean([r['overall']     for r in breakdown_rows]):>8.1%}  "
          f"{_mean([r['schema']      for r in breakdown_rows]):>7.1%}  "
          f"{_mean([r['csr_frac']    for r in breakdown_rows]):>8.1%}  "
          f"{sum(r['csr_bin'] for r in breakdown_rows)}/{n} pass  "
          f"{_mean([r['tool_acc']    for r in breakdown_rows]):>6.1%}  "
          f"{_mean([r['consistency'] for r in breakdown_rows]):>6.1%}")
print(f"{'='*105}")

# ── Failure summary ────────────────────────────────────────────────────────
print("\nFAILURE ANALYSIS SUMMARY")
print("-" * 55)
all_ftypes = {}
for row in breakdown_rows:
    for ft in row["ftypes"]:
        all_ftypes[ft] = all_ftypes.get(ft, 0) + 1

if all_ftypes:
    for ft, cnt in sorted(all_ftypes.items(), key=lambda x: -x[1]):
        tasks_with = [r["task_id"] for r in breakdown_rows if ft in r["ftypes"]]
        print(f"  {ft:<35}  {cnt}×  ({', '.join(tasks_with)})")
else:
    print("  No constraint failures — all T1 tasks passed ✓")

# ── Key finding ────────────────────────────────────────────────────────────
fail_rows = [r for r in breakdown_rows if not r["csr_bin"]]
pass_rows = [r for r in breakdown_rows if r["csr_bin"]]
print(f"\n  Binary CSR: {len(pass_rows)}/{n} pass,  {len(fail_rows)}/{n} fail")
if pass_rows: print(f"  Mean overall (passing): {_mean([r['overall'] for r in pass_rows]):.1%}")
if fail_rows: print(f"  Mean overall (failing): {_mean([r['overall'] for r in fail_rows]):.1%}")
if len(fail_rows) == 1:
    fr = fail_rows[0]
    print(f"\n  ⚠  Sole binary failure: {fr['task_id']}")
    print(f"     Failure type: {', '.join(set(fr['ftypes']))}")
    print(f"     → T1 mean depressed by a SINGLE API metadata gap, not")
    print(f"       a systematic single-turn extraction weakness.")

# Save
out_path = Path(EVAL_DIR) / "t1_breakdown.json"
out_path.write_text(json.dumps(breakdown_rows, indent=2, ensure_ascii=False))
print(f"\nSaved → {out_path}")